In [7]:
import pandas as pd

In [8]:
#slash error verdi deye doubleladim
tickets = pd.read_csv("C:\\Users\\aysum\\m1-06-lab-loading-cleaning\\data_safe_copy.csv")
tickets

,ticket_id,opened_at,category,priority,resolution_minutes
0,TK-0001,2024-03-01 09:00,billing,low,45
1,TK-0002,2024-03-01 09:30,account,medium,30
2,TK-0003,2024-03-01 10:00,technical,high,75
3,TK-0004,2024-03-01 10:30,billing,low,NaN
4,TK-0005,2024-03-01 11:00,account,medium,unknown
...,...,...,...,...,...
495,TK-0496,2024-03-01 11:30,TECHNICAL,high,60
496,TK-0497,2024-03-01 12:00,billing,high,15
497,TK-0498,2024-03-01 12:30,account,low,90
498,TK-0499,2024-03-01 13:00,technical,medium,120


In [9]:
#sayi bilmek ucun
print("Number of raws: ",tickets.shape[0])

Number of raws:  500


In [10]:
tickets.head()
tickets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   ticket_id           500 non-null    object
 1   opened_at           500 non-null    object
 2   category            500 non-null    object
 3   priority            500 non-null    object
 4   resolution_minutes  450 non-null    object
dtypes: object(5)
memory usage: 19.7+ KB


In [11]:
# 'opened_at' sütununu datetime formatına çeviririk, səhv dəyərlər NaT olacaq
tickets['opened_at'] = pd.to_datetime(tickets['opened_at'], errors='coerce')

# 'category' sütunundakı boşluqları silirik və hərfləri kiçik edirik
tickets['category'] = tickets['category'].str.strip().str.lower()

# 'priority' sütunundakı boşluqları silirik və hərfləri kiçik edirik
tickets['priority'] = tickets['priority'].str.strip().str.lower()

# 'category' sütunundakı unikal dəyərləri çap edirik
print("Unique values for category:", tickets['category'].unique())

# 'priority' sütunundakı unikal dəyərləri çap edirik
print("Unique values for priority:", tickets['priority'].unique())

Unique values for category: ['billing' 'account' 'technical']
Unique values for priority: ['low' 'medium' 'high']


In [12]:
# 'resolution_minutes' sütununu rəqəmə çeviririk, səhv dəyərlər NaN olacaq
tickets['resolution_minutes'] = pd.to_numeric(tickets['resolution_minutes'], errors='coerce')

# 'resolution_minutes' sütununda çatışmayan dəyərlərin sayını hesablayırıq
missing_count = tickets['resolution_minutes'].isna().sum()
print("Number of missing resolution_minutes:", missing_count)

# Çatışmayan dəyərləri olan sətirləri göstəririk
tickets[tickets['resolution_minutes'].isna()]

Number of missing resolution_minutes: 100


,ticket_id,opened_at,category,priority,resolution_minutes
3,TK-0004,2024-03-01 10:30:00,billing,low,NaN
4,TK-0005,2024-03-01 11:00:00,account,medium,NaN
13,TK-0014,2024-03-01 10:30:00,billing,low,NaN
14,TK-0015,2024-03-01 11:00:00,account,medium,NaN
23,TK-0024,2024-03-01 10:30:00,billing,low,NaN
...,...,...,...,...,...
474,TK-0475,2024-03-01 11:00:00,account,medium,NaN
483,TK-0484,2024-03-01 10:30:00,billing,low,NaN
484,TK-0485,2024-03-01 11:00:00,account,medium,NaN
493,TK-0494,2024-03-01 10:30:00,billing,low,NaN


In [13]:
# Ümumi sətir sayını hesablayırıq
total_records = tickets.shape[0]
print("Total records before cleaning:", total_records)

# resolution_minutes sütununda NaN olan sətirləri silirik
tickets_clean = tickets.dropna(subset=['resolution_minutes'])

# Qalan sətirlərdə resolution_minutes mənfi deyilmi yoxlayırıq
if (tickets_clean['resolution_minutes'] >= 0).all():
    print("All resolution_minutes are non-negative.")
else:
    print("Some resolution_minutes are negative!!!.")

# Kateqoriyalar və prioritetlərdə əlavə boşluq və ya fərqli böyük/kiçik hərfləri normalizə edirik
categories_normalized = tickets_clean['category'].str.strip().str.lower().unique()
priorities_normalized = tickets_clean['priority'].str.strip().str.lower().unique()

print("Unique categories after cleaning:", categories_normalized)
print("Unique priorities after cleaning:", priorities_normalized)

Total records before cleaning: 500
All resolution_minutes are non-negative.
Unique categories after cleaning: ['billing' 'account' 'technical']
Unique priorities after cleaning: ['low' 'medium' 'high']


In [14]:
# Təmizləmə əməliyyatından əvvəl və sonra sətir sayını müqayisə edirik
clean_records = tickets_clean.shape[0]

print("Before cleaning of dataset number of row :", total_records)
print("After cleaning of dataset number of row : ", clean_records)

Before cleaning of dataset number of row : 500
After cleaning of dataset number of row :  400


In [15]:
# 'resolution_minutes' sütununda çatışmayan dəyərlərin sayını hesablayırıq
missing_resolution = tickets['resolution_minutes'].isna().sum()

# Təmizlənmiş dataset üzrə orta həll müddətini hesablayırıq
avg_resolution = tickets_clean['resolution_minutes'].mean()

# Təmizlənmiş dataset üzrə maksimum həll müddətini tapırıq
max_resolution = tickets_clean['resolution_minutes'].max()

# Bütün statistikaları bir dictionary-də saxlayırıq
summary = {
    "total_records": total_records,
    "clean_records": clean_records,
    "missing_resolution": missing_resolution,
    "avg_resolution": avg_resolution,
    "max_resolution": max_resolution
}

# Məlumat keyfiyyəti icmalını çap edirik
print("Data Quality Summary:")
for key, value in summary.items():
    print(f"{key}: {value}")

Data Quality Summary:
total_records: 500
clean_records: 400
missing_resolution: 100
avg_resolution: 57.5
max_resolution: 120.0


In [17]:
# Təhlil nəticələrinin düzgünlüyünü yoxlayırıq
# Təmizlənmiş sətrlərin sayı + çatışmayan dəyərlərin sayı ümumi sətr sayına bərabər olmalıdır
if clean_records + missing_resolution == total_records:
    print("Summary check passed: clean_records + missing_resolution equals total_records.")
else:
    print("Summary check failed!")

Summary check passed: clean_records + missing_resolution equals total_records.
